# 🔍 VERIFIKASI PESERTA APPROVED
## Cross-Check Form Keikutsertaan dengan Certiport

---
**Deskripsi Program:**
- Memproses data dari Form Keikutsertaan Sertifikasi Kompetensi
- Cross-check dengan database Certiport untuk verifikasi peserta
- **TUJUAN**: Mencari peserta yang mengklaim sudah sertifikasi tapi TIDAK ADA di Certiport
- Menghasilkan daftar peserta yang perlu verifikasi manual:
  - ✅ **TERVERIFIKASI** - Ada di database Certiport
  - ❌ **TIDAK DITEMUKAN** - Klaim sertifikasi tapi tidak ada di Certiport (PERLU DICEK!)

**Input Files:**
1. `FORM KEIKUTSERTAAN SERTIFIKASI KOMPETENSI.csv` - Data form keikutsertaan
2. `certiport.csv` - Database hasil ujian dari Certiport

**Output Files:**
- `VERIFIKASI_APPROVED.xlsx` - File Excel dengan hasil verifikasi

---

## 1️⃣ Import Library & Setup

In [13]:
# ============================================================
# IMPORT LIBRARY YANG DIPERLUKAN
# ============================================================
import pandas as pd
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import warnings
warnings.filterwarnings('ignore')

print("✅ Library berhasil diimport!")
print("   - pandas: untuk manipulasi data")
print("   - fuzzywuzzy: untuk fuzzy string matching")

✅ Library berhasil diimport!
   - pandas: untuk manipulasi data
   - fuzzywuzzy: untuk fuzzy string matching


## 2️⃣ Load Data

In [14]:
# ============================================================
# LOAD DATA FORM KEIKUTSERTAAN
# ============================================================
print(f"{'='*70}")
print("📋 LOAD DATA FORM KEIKUTSERTAAN SERTIFIKASI")
print(f"{'='*70}")

# Load data form keikutsertaan (dari folder parent)
form_df = pd.read_csv('../FORM KEIKUTSERTAAN SERTIFIKASI KOMPETENSI.csv')

print(f"\n✅ Data Form Keikutsertaan berhasil diload!")
print(f"   Total peserta: {len(form_df)}")

# Tampilkan kolom yang tersedia
print(f"\n📋 Kolom yang tersedia:")
for i, col in enumerate(form_df.columns, 1):
    print(f"   {i:>2}. {col}")

📋 LOAD DATA FORM KEIKUTSERTAAN SERTIFIKASI

✅ Data Form Keikutsertaan berhasil diload!
   Total peserta: 159

📋 Kolom yang tersedia:
    1. ID
    2. Start time
    3. Completion time
    4. Email
    5. Name
    6. Last modified time
    7. NIM (Nomor Induk Mahasiswa)
    8. NAMA LENGKAP
    9. NOMOR HANDHONE
   10. Email2
   11. PROGRAM STUDI
   12. PILIH PROGRAM SERTIFIKASI YANG SUDAH PERNAH DIAMBIL
   13. PILIH PROGRAM SERTIFIKASI YANG SUDAH PERNAH DIAMBIL2
   14. PILIH SUBPROGRAM MOS
   15. PILIH SUBPROGRAM MCF
   16. PILIH SUBPROGRAM
   17. Dengan ini saya menyatakan bahwa data yang saya masukkan adalah BENAR, yang dapat DIPERTANGGUNG JAWABKAN dan jika saya melanggar ketentuan, maka saya bersedia mendapatkan SANKSI sesuai aturan dari...
   18. Catatan: Surat Keterangan Bukti Peserta Sertifikasi diberikan setelah ITCC melakukan validasi data peserta dan akan diterbitkan secepatnya, dan dikirim ke email institusi/email kampus IT-PLN peserta.



In [15]:
# ============================================================
# LOAD DATA CERTIPORT
# ============================================================
print(f"{'='*70}")
print("📊 LOAD DATA CERTIPORT")
print(f"{'='*70}")

# Load data Certiport (dari folder parent)
certiport_df = pd.read_csv('../certiport.csv', skiprows=4)
certiport_df = certiport_df.dropna(axis=1, how='all')

# Filter berdasarkan jenis ujian
certiport_mcf = certiport_df[certiport_df['Exam'].str.contains('AI-900', na=False)].copy()
certiport_mos = certiport_df[certiport_df['Exam'].str.contains('Office 2019', na=False)].copy()

print(f"\n✅ Data Certiport berhasil diload!")
print(f"   Total record    : {len(certiport_df)}")
print(f"   MCF (AI-900)    : {len(certiport_mcf)}")
print(f"   MOS (Office 2019): {len(certiport_mos)}")

# Breakdown Exam
print(f"\n📊 Breakdown Exam di Certiport:")
print(certiport_df['Exam'].value_counts().to_string())

📊 LOAD DATA CERTIPORT

✅ Data Certiport berhasil diload!
   Total record    : 4853
   MCF (AI-900)    : 897
   MOS (Office 2019): 3445

📊 Breakdown Exam di Certiport:
Exam
Microsoft Word (Office 2019)                                         2714
AI-900: Microsoft Azure AI Fundamentals                               897
Microsoft Excel (Office 2019)                                         624
Microsoft Word (Office 2016)                                          287
Microsoft Excel (Office 2016)                                         161
Microsoft PowerPoint (Office 2019)                                    104
Microsoft PowerPoint (Office 2016)                                     53
Microsoft Word Expert (Office 2019)                                     3
Microsoft Excel (Microsoft 365 Apps)                                    2
Microsoft Word (Microsoft 365 Apps)                                     1
SC-900: Microsoft Security, Compliance, and Identity Fundamentals       1


## 3️⃣ Siapkan & Bersihkan Data

In [16]:
# ============================================================
# SIAPKAN DAN BERSIHKAN DATA FORM
# ============================================================
print(f"{'='*70}")
print("🔧 PERSIAPAN DATA FORM KEIKUTSERTAAN")
print(f"{'='*70}")

# Bersihkan nama dan NIM
form_df['NAMA LENGKAP'] = form_df['NAMA LENGKAP'].fillna('').str.strip().str.upper()
form_df['NIM (Nomor Induk Mahasiswa)'] = form_df['NIM (Nomor Induk Mahasiswa)'].astype(str).str.strip()

# Buat kolom Email Final (prioritas Email, jika kosong/anonymous gunakan Email2)
def get_email_final(row):
    email = str(row['Email']) if pd.notna(row['Email']) else ''
    email2 = str(row['Email2']) if pd.notna(row['Email2']) else ''
    
    if email and email != 'anonymous' and email.strip() != '' and email.strip().lower() != 'nan':
        return email.strip()
    elif email2 and email2.strip() != '' and email2.strip().lower() != 'nan':
        return email2.strip()
    else:
        return '-'

form_df['Email Final'] = form_df.apply(get_email_final, axis=1)

# Tampilkan preview data
print(f"\n✅ Data berhasil diproses!")
print(f"\n📋 Preview Data (5 baris pertama):")
preview_cols = ['NAMA LENGKAP', 'NIM (Nomor Induk Mahasiswa)', 'Email Final']
print(form_df[preview_cols].head().to_string(index=False))

🔧 PERSIAPAN DATA FORM KEIKUTSERTAAN

✅ Data berhasil diproses!

📋 Preview Data (5 baris pertama):
          NAMA LENGKAP NIM (Nomor Induk Mahasiswa)                   Email Final
    RATU ADISYA FAMELA                   202231105       ratu2231105@itpln.ac.id
MUHAMMAD RAIHAN AZZAKY                   202241017     raihan2241017@itpln.ac.id
     VIA ISNATUL LAILA                   202241013 viaisnatul2241013@itpln.ac.id
  NAUFAL YURFANA AISAL                   202241012     naufal2241012@itpln.ac.id
  WIDY SYAFITRI SLAMET                   202231070       Widy2231070@itpln.ac.id


## 4️⃣ Fungsi Fuzzy Matching (Improved)

In [17]:
# ============================================================
# FUNGSI-FUNGSI UNTUK FUZZY MATCHING (IMPROVED)
# ============================================================

def normalize_name(name):
    """Normalisasi nama: uppercase, hapus karakter khusus kecuali - dan ., rapikan spasi"""
    if pd.isna(name):
        return ""
    # Keep alphanumeric, spaces, dash, and dot
    cleaned = ''.join(c for c in str(name) if c.isalnum() or c.isspace() or c in '.-')
    return ' '.join(cleaned.upper().strip().split())

def reverse_name(name):
    """Balik urutan nama (untuk antisipasi nama terbalik)"""
    parts = name.split()
    if len(parts) >= 2:
        return ' '.join(parts[::-1])
    return name

def count_matching_words(name1, name2, min_word_length=2):
    """Hitung jumlah kata yang sama antara dua nama"""
    words1 = set(w for w in name1.split() if len(w) >= min_word_length)
    words2 = set(w for w in name2.split() if len(w) >= min_word_length)
    return len(words1.intersection(words2))

def get_min_matching_words(name):
    """
    Tentukan minimal kata yang harus cocok berdasarkan jumlah kata dalam nama:
    - 4+ kata: minimal 3 kata cocok
    - 3 kata: minimal 2 kata cocok
    - 2 kata: minimal 2 kata cocok
    - 1 kata: harus ada tanda - atau . di nama (special case)
    """
    words = [w for w in name.split() if len(w) >= 2]
    word_count = len(words)
    
    if word_count >= 4:
        return 3
    elif word_count == 3:
        return 2
    elif word_count == 2:
        return 2
    else:
        return 1  # Special handling for single word names

def is_single_word_match(name1, name2):
    """
    Untuk nama 1 kata, cek apakah ada tanda - atau . yang menunjukkan kecocokan
    Contoh: 'ABDUL-RAHMAN' atau 'A.RAHMAN'
    """
    # Cek apakah nama mengandung - atau .
    has_special_char = '-' in name1 or '.' in name1 or '-' in name2 or '.' in name2
    if not has_special_char:
        return False
    
    # Normalize tanpa - dan . untuk perbandingan
    clean1 = name1.replace('-', ' ').replace('.', ' ').upper()
    clean2 = name2.replace('-', ' ').replace('.', ' ').upper()
    
    # Cek kecocokan
    score = fuzz.token_set_ratio(clean1, clean2)
    return score >= 85

def find_best_match(peserta_name, certiport_names, threshold=80):
    """
    Mencari kecocokan nama terbaik dengan aturan:
    - 4+ kata: minimal 3 kata cocok
    - 3 kata: minimal 2 kata cocok
    - 2 kata: minimal 2 kata cocok
    - 1 kata: harus ada tanda - atau . dan match
    """
    peserta_normalized = normalize_name(peserta_name)
    min_words_required = get_min_matching_words(peserta_normalized)
    peserta_word_count = len([w for w in peserta_normalized.split() if len(w) >= 2])
    
    # Cek exact match
    if peserta_normalized in certiport_names:
        return peserta_normalized, 100, "Exact Match"
    
    # Cek nama terbalik
    reversed_name_str = reverse_name(peserta_normalized)
    if reversed_name_str in certiport_names:
        return reversed_name_str, 100, "Exact Match (Nama Terbalik)"
    
    best_score = 0
    best_match = None
    match_type = ""
    best_word_count = 0
    
    for cert_name in certiport_names:
        # Special handling untuk nama 1 kata
        if peserta_word_count == 1:
            if is_single_word_match(peserta_normalized, cert_name):
                return cert_name, 90, "Single Word Match (dengan tanda - atau .)"
            continue
        
        matching_words = count_matching_words(peserta_normalized, cert_name)
        
        # Cek minimum kata yang harus cocok
        if matching_words < min_words_required:
            continue
        
        # Berbagai metode fuzzy matching
        score1 = fuzz.ratio(peserta_normalized, cert_name)
        score2 = fuzz.token_set_ratio(peserta_normalized, cert_name)
        score3 = fuzz.token_sort_ratio(peserta_normalized, cert_name)
        score4 = fuzz.partial_ratio(peserta_normalized, cert_name)
        
        max_score = max(score1, score2, score3, score4)
        
        if matching_words > best_word_count or (matching_words == best_word_count and max_score > best_score):
            best_score = max_score
            best_match = cert_name
            best_word_count = matching_words
            if max_score == score1:
                match_type = "Fuzzy Ratio"
            elif max_score == score2:
                match_type = "Token Set Ratio"
            elif max_score == score3:
                match_type = "Token Sort Ratio"
            else:
                match_type = "Partial Ratio"
    
    if best_score >= threshold and best_word_count >= min_words_required:
        return best_match, best_score, f"{match_type} ({best_word_count}/{min_words_required} kata cocok)"
    
    return None, best_score, "Tidak Ditemukan"

print("✅ Fungsi matching (IMPROVED) berhasil dibuat!")
print("\n📋 Aturan Minimal Kata Cocok:")
print("   • Nama 4+ kata → minimal 3 kata cocok")
print("   • Nama 3 kata  → minimal 2 kata cocok")
print("   • Nama 2 kata  → minimal 2 kata cocok")
print("   • Nama 1 kata  → harus ada tanda - atau . dan match")

✅ Fungsi matching (IMPROVED) berhasil dibuat!

📋 Aturan Minimal Kata Cocok:
   • Nama 4+ kata → minimal 3 kata cocok
   • Nama 3 kata  → minimal 2 kata cocok
   • Nama 2 kata  → minimal 2 kata cocok
   • Nama 1 kata  → harus ada tanda - atau . dan match


## 5️⃣ Proses Verifikasi dengan Certiport

In [18]:
# ============================================================
# PROSES VERIFIKASI FORM DENGAN CERTIPORT
# ============================================================
print(f"{'='*70}")
print("🔍 PROSES VERIFIKASI PESERTA DENGAN CERTIPORT")
print(f"{'='*70}")

# Buat Full Name dan normalisasi untuk Certiport
certiport_mcf['Full Name'] = certiport_mcf['First Name'].fillna('') + ' ' + certiport_mcf['Last Name'].fillna('')
certiport_mos['Full Name'] = certiport_mos['First Name'].fillna('') + ' ' + certiport_mos['Last Name'].fillna('')

certiport_mcf_names = [normalize_name(name) for name in certiport_mcf['Full Name'].tolist()]
certiport_mos_names = [normalize_name(name) for name in certiport_mos['Full Name'].tolist()]

# Gabungkan semua nama certiport
all_certiport_names = list(set(certiport_mcf_names + certiport_mos_names))

# Proses verifikasi
verifikasi_results = []

print(f"\n⏳ Memproses {len(form_df)} peserta...")

for idx, row in form_df.iterrows():
    nama = row['NAMA LENGKAP']
    nim = row['NIM (Nomor Induk Mahasiswa)']
    no_hp = str(row.get('NOMOR HANDHONE', '')) if pd.notna(row.get('NOMOR HANDHONE')) else '-'
    email_final = row['Email Final']
    prodi = row.get('PROGRAM STUDI', '-')
    
    # Ambil subprogram yang di-REQUEST dari form
    subprog_mos = str(row.get('PILIH SUBPROGRAM MOS', '')) if pd.notna(row.get('PILIH SUBPROGRAM MOS')) else ''
    subprog_mcf = str(row.get('PILIH SUBPROGRAM MCF', '')) if pd.notna(row.get('PILIH SUBPROGRAM MCF')) else ''
    subprog_both = str(row.get('PILIH SUBPROGRAM', '')) if pd.notna(row.get('PILIH SUBPROGRAM')) else ''
    
    # Gabungkan subprogram
    subprogram_list = []
    if subprog_mos and subprog_mos.strip() and subprog_mos.lower() != 'nan':
        subprogram_list.append(subprog_mos.strip())
    if subprog_mcf and subprog_mcf.strip() and subprog_mcf.lower() != 'nan':
        subprogram_list.append(subprog_mcf.strip())
    if subprog_both and subprog_both.strip() and subprog_both.lower() != 'nan':
        subprogram_list.append(subprog_both.strip())
    subprogram_display = ' & '.join(subprogram_list) if subprogram_list else '-'
    
    # Cek di KEDUA database Certiport
    match_mos, score_mos, match_type_mos = find_best_match(nama, certiport_mos_names)
    match_mcf, score_mcf, match_type_mcf = find_best_match(nama, certiport_mcf_names)
    
    # Tentukan status verifikasi
    if match_mos and match_mcf:
        status = '✅ TERVERIFIKASI'
        keterangan = 'Ditemukan di MOS & MCF'
        nama_certiport = f"MOS: {match_mos} | MCF: {match_mcf}"
        ada_di_certiport = True
    elif match_mos:
        status = '✅ TERVERIFIKASI'
        keterangan = 'Ditemukan di MOS'
        nama_certiport = match_mos
        ada_di_certiport = True
    elif match_mcf:
        status = '✅ TERVERIFIKASI'
        keterangan = 'Ditemukan di MCF'
        nama_certiport = match_mcf
        ada_di_certiport = True
    else:
        status = '❌ TIDAK DITEMUKAN'
        keterangan = 'TIDAK ADA di database Certiport - PERLU VERIFIKASI MANUAL!'
        nama_certiport = '-'
        ada_di_certiport = False
    
    verifikasi_results.append({
        'Nama Lengkap': nama,
        'NIM': nim,
        'No HP': no_hp,
        'Email Final': email_final,
        'Program Studi': prodi,
        'Subprogram Request': subprogram_display,
        'Nama di Certiport': nama_certiport,
        'Skor Kecocokan': max(score_mos, score_mcf),
        'Status': status,
        'Keterangan': keterangan,
        'Ada di Certiport': ada_di_certiport
    })

# Buat DataFrame hasil
verifikasi_df = pd.DataFrame(verifikasi_results)

# Pisahkan hasil
terverifikasi_df = verifikasi_df[verifikasi_df['Ada di Certiport'] == True].copy()
tidak_ditemukan_df = verifikasi_df[verifikasi_df['Ada di Certiport'] == False].copy()

print(f"\n✅ Proses verifikasi selesai!")
print(f"\n{'='*70}")
print("📊 HASIL VERIFIKASI")
print(f"{'='*70}")
print(f"   ✅ TERVERIFIKASI (Ada di Certiport)     : {len(terverifikasi_df):>4} peserta")
print(f"   ❌ TIDAK DITEMUKAN (Tidak di Certiport) : {len(tidak_ditemukan_df):>4} peserta")
print(f"   {'─'*45}")
print(f"   TOTAL                                   : {len(verifikasi_df):>4} peserta")

🔍 PROSES VERIFIKASI PESERTA DENGAN CERTIPORT

⏳ Memproses 159 peserta...

✅ Proses verifikasi selesai!

📊 HASIL VERIFIKASI
   ✅ TERVERIFIKASI (Ada di Certiport)     :  127 peserta
   ❌ TIDAK DITEMUKAN (Tidak di Certiport) :   32 peserta
   ─────────────────────────────────────────────
   TOTAL                                   :  159 peserta


## 6️⃣ Tampilkan Peserta TIDAK DITEMUKAN

In [19]:
# ============================================================
# TAMPILKAN PESERTA TIDAK DITEMUKAN DI CERTIPORT
# ============================================================
print(f"{'='*90}")
print("❌ PESERTA YANG TIDAK DITEMUKAN DI CERTIPORT (PERLU VERIFIKASI MANUAL!)")
print(f"{'='*90}")
print(f"Total: {len(tidak_ditemukan_df)} peserta\n")

if len(tidak_ditemukan_df) > 0:
    display_cols = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Subprogram Request']
    print(tidak_ditemukan_df[display_cols].to_string(index=False))
else:
    print("✅ Semua peserta terverifikasi ada di Certiport!")

❌ PESERTA YANG TIDAK DITEMUKAN DI CERTIPORT (PERLU VERIFIKASI MANUAL!)
Total: 32 peserta

                                Nama Lengkap       NIM         No HP                                       Email Final     Subprogram Request
                          JAPAR SIRINGORINGO 202214045   85212078991                          japar2214045@itpln.ac.id  MOS: Office Word 2019
                          JAPAR SIRINGORINGO 202214045   85212078991                          japar2214045@itpln.ac.id  MOS: Office Word 2019
                  LALU MUHAMMAD RISGAN NAZWA 202231009 6285973913275                           Lalu2231009@itpln.ac.id  MOS: Office Word 2019
            RITCHIE CHORINUS TIOLUNG MAABUAT 202211088 6282271005326                        Ritchie2211088@itpln.ac.id MOS: Office Excel 2019
                     ANNA FINCE MARIANA WOUW 202111062  852116359972                           anna2111062@itpln.ac.id  MOS: Office Word 2019
                        HARTANTO TRI SUBEKTI 202214010 628

## 7️⃣ Export ke Excel

In [20]:
# ============================================================
# EXPORT KE EXCEL DENGAN FORMATTING
# ============================================================
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

print(f"{'='*70}")
print("📊 EXPORT KE EXCEL")
print(f"{'='*70}")

wb = Workbook()

# Style definitions
header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)
terverifikasi_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")  # Hijau
tidak_ditemukan_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")  # Merah
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

def style_worksheet(ws):
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center')
        cell.border = thin_border
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.border = thin_border
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 50)

# Sheet 1: Dashboard Summary
ws1 = wb.active
ws1.title = "Dashboard Summary"
summary = [
    ["KATEGORI", "JUMLAH", "PERSENTASE", "KETERANGAN"],
    ["Total Peserta Form", len(verifikasi_df), "100%", "ℹ️"],
    ["", "", "", ""],
    ["TERVERIFIKASI", len(terverifikasi_df), f"{len(terverifikasi_df)/len(verifikasi_df)*100:.1f}%" if len(verifikasi_df) > 0 else "0%", "✅ Ada di Certiport"],
    ["TIDAK DITEMUKAN", len(tidak_ditemukan_df), f"{len(tidak_ditemukan_df)/len(verifikasi_df)*100:.1f}%" if len(verifikasi_df) > 0 else "0%", "❌ Perlu verifikasi manual"],
]
for row in summary:
    ws1.append(row)
style_worksheet(ws1)

# Sheet 2: TIDAK DITEMUKAN (Perlu Verifikasi)
ws2 = wb.create_sheet("Tidak Ditemukan")
cols_tidak = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Program Studi', 'Subprogram Request', 'Keterangan']
if len(tidak_ditemukan_df) > 0:
    for r in dataframe_to_rows(tidak_ditemukan_df[cols_tidak], index=False, header=True):
        ws2.append(r)
else:
    ws2.append(cols_tidak)
    ws2.append(["Tidak ada data"] + [""]*(len(cols_tidak)-1))
style_worksheet(ws2)
# Warnai merah
for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
    for cell in row:
        cell.fill = tidak_ditemukan_fill

# Sheet 3: TERVERIFIKASI
ws3 = wb.create_sheet("Terverifikasi")
cols_ver = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Program Studi', 'Subprogram Request', 'Nama di Certiport', 'Keterangan']
if len(terverifikasi_df) > 0:
    for r in dataframe_to_rows(terverifikasi_df[cols_ver], index=False, header=True):
        ws3.append(r)
else:
    ws3.append(cols_ver)
    ws3.append(["Tidak ada data"] + [""]*(len(cols_ver)-1))
style_worksheet(ws3)
# Warnai hijau
for row in ws3.iter_rows(min_row=2, max_row=ws3.max_row):
    for cell in row:
        cell.fill = terverifikasi_fill

# Sheet 4: Semua Hasil
ws4 = wb.create_sheet("Semua Hasil")
cols_all = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Program Studi', 'Subprogram Request', 'Nama di Certiport', 'Status', 'Keterangan']
for r in dataframe_to_rows(verifikasi_df[cols_all], index=False, header=True):
    ws4.append(r)
style_worksheet(ws4)
# Warnai berdasarkan status
for row_idx, row in enumerate(ws4.iter_rows(min_row=2, max_row=ws4.max_row), start=2):
    status = str(ws4.cell(row=row_idx, column=8).value)  # Kolom Status
    fill = terverifikasi_fill if 'TERVERIFIKASI' in status else tidak_ditemukan_fill
    for cell in row:
        cell.fill = fill

# Simpan file (overwrite jika sudah ada)
excel_file = 'VERIFIKASI_APPROVED.xlsx'
wb.save(excel_file)

print(f"\n✅ File Excel berhasil disimpan: {excel_file}")
print(f"\n📋 DAFTAR SHEETS:")
print(f"   1. Dashboard Summary  - Ringkasan keseluruhan")
print(f"   2. Tidak Ditemukan    - 🔴 {len(tidak_ditemukan_df)} peserta (PERLU VERIFIKASI MANUAL!)")
print(f"   3. Terverifikasi      - 🟢 {len(terverifikasi_df)} peserta (Ada di Certiport)")
print(f"   4. Semua Hasil        - Detail lengkap semua peserta")
print(f"\n📌 KETERANGAN WARNA:")
print(f"   🟢 Hijau = Terverifikasi (Ada di Certiport)")
print(f"   🔴 Merah = Tidak Ditemukan (Perlu verifikasi manual)")

📊 EXPORT KE EXCEL

✅ File Excel berhasil disimpan: VERIFIKASI_APPROVED.xlsx

📋 DAFTAR SHEETS:
   1. Dashboard Summary  - Ringkasan keseluruhan
   2. Tidak Ditemukan    - 🔴 32 peserta (PERLU VERIFIKASI MANUAL!)
   3. Terverifikasi      - 🟢 127 peserta (Ada di Certiport)
   4. Semua Hasil        - Detail lengkap semua peserta

📌 KETERANGAN WARNA:
   🟢 Hijau = Terverifikasi (Ada di Certiport)
   🔴 Merah = Tidak Ditemukan (Perlu verifikasi manual)


## 🔍 Fungsi Pencarian Manual

In [21]:
# ============================================================
# FUNGSI PENCARIAN MANUAL
# ============================================================

def cari_nama(nama):
    """
    Fungsi untuk mencari nama secara manual di database Certiport
    
    Parameter:
    - nama: nama yang ingin dicari
    
    Contoh: cari_nama('JOHN DOE')
    """
    match_mos, score_mos, match_type_mos = find_best_match(nama, certiport_mos_names)
    match_mcf, score_mcf, match_type_mcf = find_best_match(nama, certiport_mcf_names)
    
    print(f"\n🔍 Hasil Pencarian untuk: {nama}")
    print(f"   {'='*70}")
    
    print(f"\n   📋 HASIL CEK MOS (Office 2019):")
    if match_mos:
        print(f"      ✅ DITEMUKAN")
        print(f"      Nama di Certiport: {match_mos}")
        print(f"      Skor Kecocokan: {score_mos}")
        print(f"      Tipe Match: {match_type_mos}")
    else:
        print(f"      ❌ TIDAK DITEMUKAN")
        print(f"      Skor Tertinggi: {score_mos}")
    
    print(f"\n   📋 HASIL CEK MCF (Azure AI-900):")
    if match_mcf:
        print(f"      ✅ DITEMUKAN")
        print(f"      Nama di Certiport: {match_mcf}")
        print(f"      Skor Kecocokan: {score_mcf}")
        print(f"      Tipe Match: {match_type_mcf}")
    else:
        print(f"      ❌ TIDAK DITEMUKAN")
        print(f"      Skor Tertinggi: {score_mcf}")

print("✅ Fungsi cari_nama() siap digunakan!")
print("\n📖 Cara Pakai:")
print("   cari_nama('NAMA LENGKAP')")

✅ Fungsi cari_nama() siap digunakan!

📖 Cara Pakai:
   cari_nama('NAMA LENGKAP')
